In [6]:
import pandas as pd
import statsmodels.formula.api as smf

df = pd.read_csv('physics_adjusted_dataset.csv')
df = df.dropna(subset=['CorrectedLapTime', 'GapToCarAhead', 'Compound', 'Circuit', 'Season'])

df['IsPost2022'] = (df['Season'] >= 2022).astype(int)
df['Interaction'] = df['GapToCarAhead'] * df['IsPost2022']
df['RaceID'] = df['Season'].astype(str) + '_' + df['Circuit']

# some circuits only exist in one era (e.g Las Vegas, Miami are post2022 only)
# keeping only circuits present in both eras avoids Circuit/IsPost2022 collinearity
circuit_era_counts = df.groupby('Circuit')['IsPost2022'].nunique()
both_era_circuits = circuit_era_counts[circuit_era_counts == 2].index
filtered_df = df[df['Circuit'].isin(both_era_circuits)]

formula = 'CorrectedLapTime ~ GapToCarAhead + IsPost2022 + Interaction + C(Circuit) + C(Compound) + TyreLife + LapNumber'
model = smf.ols(formula, data=filtered_df).fit()

# clustered by race, since lap-time errors within the same race are likely correlated
model_robust = model.get_robustcov_results(cov_type='cluster', groups=filtered_df['RaceID'])
print(model_robust.summary())

                            OLS Regression Results                            
Dep. Variable:       CorrectedLapTime   R-squared:                       0.937
Model:                            OLS   Adj. R-squared:                  0.937
Method:                 Least Squares   F-statistic:                     3612.
Date:                Sun, 28 Jun 2026   Prob (F-statistic):          6.29e-102
Time:                        06:16:04   Log-Likelihood:            -1.6539e+05
No. Observations:               70618   AIC:                         3.308e+05
Df Residuals:                   70595   BIC:                         3.310e+05
Df Model:                          22                                         
Covariance Type:              cluster                                         
                                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------
Intercept           